# Review: `solver-firedrake.py` vs Labbate appendix (Eqs. 8–10, weak forms, A29)

This notebook **does not modify** `src/solver-firedrake.py`. It documents:

1. What **Eq. A29** refers to in `docs/Labbate_APAM9301_Final_06032026.tex` (appendix weak form with total-pressure KBM).
2. What the **code actually implements** (Plan A: effective $D$ coefficient, Saarelma–Connor $\alpha$-based $D_{\mathrm{KBM}}$).
3. Automated / semi-automated checks you can re-run after edits.

Run from `tests/firedrake_solver/`. Firedrake tests are skipped if Firedrake is not importable.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
from numpy.testing import assert_allclose, assert_array_less

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

FD_PATH = ROOT / "src" / "solver-firedrake.py"
fd_src = FD_PATH.read_text()

_fd_spec = importlib.util.spec_from_file_location("solver_firedrake", FD_PATH)
_fd_mod = importlib.util.module_from_spec(_fd_spec)
_fd_spec.loader.exec_module(_fd_mod)
saarelma_connor_firedrake = _fd_mod.saarelma_connor_firedrake

try:
    from firedrake import IntervalMesh, FunctionSpace, Function, TestFunction, dx, Constant
    _HAS_FD = True
except Exception as _e:
    _HAS_FD = False
    _FD_IMPORT_ERR = _e

print("Firedrake available:", _HAS_FD)
if not _HAS_FD:
    print("  (Firedrake-only cells will be skipped)")
    print(" ", _FD_IMPORT_ERR)

## 1. Equation map (Labbate ↔ code)

| Item | Labbate label | Strong form | Code (`_build_f1` / `F2` / `F3`) |
|------|---------------|-------------|----------------------------------|
| Electron transport | **Eq. (8)** | $\partial_x[\langle\|\nabla r\|^2\rangle D_{\mathrm{ped}} \partial_x n_e] = -n_e S_i(\langle n_{FC}\rangle+\langle n_{CX}\rangle)$ | `F1` with $D_{\mathrm{ped}} \to D_{\mathrm{NEO}}+D_{\mathrm{KBM}}+C_{\mathrm{ETG}}/n_e$ |
| FC neutrals | **Eq. (9)** | $|V_{FC}|\partial_x[f_{FC}\langle n_{FC}\rangle\langle\|\nabla r\|^2\rangle] = n_e(S_i+S_{CX})\langle n_{FC}\rangle$ | `F2` |
| CX neutrals | **Eq. (10)** | $|V_{CX}|\partial_x[f_{CX}\langle n_{CX}\rangle\langle\|\nabla r\|^2\rangle] = n_e(\langle n_{CX}\rangle S_i - \frac12 S_{CX}\langle n_{FC}\rangle)$ | `F3` |
| Baseline weak $n_e$ | **Eq. (weak-ne)** | IBP of Eq. (8) with optional Neumann at $x_{\mathrm{inner}}$ | `F1` (+ `ds(1)` if Neumann) |
| Extended weak $n_e$ | **~A29** (appendix §“total pressure”) | Full state-dependent $D(n_e,n_e')$ with $(\partial_x n_e)^2$ terms | **Not implemented** as written — see §2 |

**Eq. A29** in the manuscript appendix is the final displayed weak form in subsection *“Rederiving electron density weak form with total pressure”* (search `sec:ne-weak-total-pressure`): find $n_e$ such that
$$
\int_{x_{\mathrm{inner}}}^0 \langle|\nabla r|^2\rangle\left[\frac{C_{ETG}}{n_e}n_e' + C_{KBM}(T_e+T_i)(n_e')^2 + \cdots\right] v_e'\,dx
- \int n_e S_i(\langle n_{FC}\rangle+\langle n_{CX}\rangle)v_e\,dx + \delta_N\,\text{(Neumann)} = 0.
$$
The **implemented** electron weak form is the simpler **Plan A** documented in the module docstring:
$$F_1 = \int g\,(D_{NEO}+D_{KBM}+C_{ETG}/n_e)\,n_e' v_e'\,dx - \int n_e S_i(n_{FC}+n_{CX}) v_e\,dx$$
which matches **Eq. (weak-ne)** only if $D_{\mathrm{ped}}$ is treated as an **effective coefficient** multiplying $n_e'$, not the fully expanded nonlinear flux in A29.

## 2. Sign / IBP check for `F1` (matches Eq. 8 + weak-ne)

Strong form: $(g D n_e')' = -n_e S_i (n_{FC}+n_{CX})$ with $g=\langle|\nabla r|^2\rangle$.

Multiply by $v_e$, integrate, IBP (boundary terms zero for Dirichlet $n_e$ at both ends):
$$\int g D n_e' v_e' = \int n_e S_i (n_{FC}+n_{CX}) v_e.$$

Code residual (volume part): `g * D * ne.dx(0) * v_e.dx(0) - ne * Si * (nFC+nCX) * v_e` → same sign.

In [ ]:
# Structural match to _build_f1_weak_form (Plan A):
#   F1 = (g * D_total * ne.dx(0) * v_e.dx(0) - ne * Si * (nFC+nCX) * v_e) * dx
# Strong Eq. (8): (g * D_ped * n_e')' = -n_e * S_i * (n_FC + n_CX)
# IBP with v_e=0 on x_inner and separatrix:  ∫ (g D n_e')' v = ∫ g D n_e' v'
assert "g_fd * flux * v_e.dx(0)" in fd_src
assert "ne * Si_fd * (nFC + nCX) * v_e" in fd_src
assert "D_total = D_NEO_fd + D_KBM_fd + C_ETG_fd / ne_for_etg" in fd_src
print("OK: F1 uses +g·D_total·n_e'·v_e' and −n_e·S_i·(n_FC+n_CX)·v_e (Eq. 8 weak form).")

## 3. `F2` / `F3` structure vs Eqs. (9)–(10)

Code uses **strong-form** Galerkin (derivative on trial function, not IBP), matching appendix §nfc-weak / §ncx-weak.

In [ ]:
def strong_residual_fc_ncx(ne, nFC, nCX, Si, Scx, g, VFC, fFC, Vcx, fCX, x):
    """Pointwise strong residuals for Eqs. (9)-(10)."""
    def ddx(f):
        return np.gradient(f, x)
    r2 = VFC * fFC * ddx(nFC * g) - ne * (Si + Scx) * nFC
    r3 = Vcx * fCX * ddx(nCX * g) - ne * (Si * nCX - 0.5 * Scx * nFC)
    return r2, r3

x = np.linspace(-0.04, 0.0, 150)
ne = 2e19 * np.ones_like(x)
nFC = 1e17 * (x - x.min()) / (x.max() - x.min())
nCX = 0.05 * nFC
Si, Scx, g = 1e-20, 5e-21, 1.0
VFC, fFC, Vcx, fCX = 1e4, 1.0, 3e3, 1.0

r2, r3 = strong_residual_fc_ncx(ne, nFC, nCX, Si, Scx, g, VFC, fFC, Vcx, fCX, x)
print("FC residual L2:", np.sqrt(np.mean(r2**2)))
print("CX residual L2:", np.sqrt(np.mean(r3**2)))
# Manufactured profiles won't satisfy PDE; this cell only sanity-checks shapes/signs of terms.
assert r2.shape == r3.shape == x.shape
print("OK: F2/F3 term structure is vectorized and same length as mesh.")

## 4. Plan A vs appendix A29 (total-pressure weak form)

| Feature | Appendix A29 (expanded) | `solver-firedrake.py` |
|---------|-------------------------|------------------------|
| $D_{\mathrm{ETG}}$ | $C_{\mathrm{ETG}}/n_e$ inside flux; gives $C_{\mathrm{ETG}}/n_e \cdot n_e'$ in volume term | Same via `C_ETG_fd / ne_for_etg` in `D_total` |
| $D_{\mathrm{KBM}}$ | $C_{\mathrm{KBM}}(T_e+T_i)\,n_e' + C_{\mathrm{KBM}} n_e (T_e+T_i)'$ → includes **$(n_e')^2$** in weak form | $D_{\mathrm{KBM}}$ from Saarelma **$\alpha$** (Eqs. 24–25), **not** $C_{\mathrm{KBM}}\partial_x(n_e T)$ inside flux |
| Weak form | Extra $C_{\mathrm{KBM}}(T_e+T_i)(n_e')^2 v_e'$ term | Missing — uses scalar/field $D_{\mathrm{KBM}}(x)$ only |
| Neumann | $\delta_N \langle\|\nabla r\|^2\rangle D\, n_e' v_e$ at $x_{\mathrm{inner}}$ | `ds(1)` term with `D_bc` at **first DOF** |

**Conclusion:** The code implements the **coupled Eqs. (8)–(10)** with the **simplified** $D_{\mathrm{ped}} = D_{\mathrm{NEO}} + D_{\mathrm{KBM}}(\alpha) + C_{\mathrm{ETG}}/n_e$ closure from Saarelma (2023), **not** the fully expanded A29 weak form with $(\partial_x n_e)^2$.

## 5. Static code review — likely bugs / inconsistencies

Findings from reading `solver-firedrake.py` (no edits applied).

In [ ]:
findings = []

# 1) Picard comment vs implementation
picard_block = fd_src[fd_src.find("for k in range(max_picard)"):
                             fd_src.find("else:", fd_src.find("for k in range(max_picard)"))]
if "updated each outer iteration" in fd_src and "_update_firedrake_ne_transport_from_ne" not in picard_block:
    findings.append(
            "HIGH: Comments say D_KBM is refreshed each Picard step, but the loop never calls "
            "_update_firedrake_ne_transport_from_ne or _set_D_KBM_from_ne_profile. D_KBM_fd stays "
            "frozen from the initial guess."
        )
# 1b) _update_firedrake_ne_transport passes parent-grid n_e into calc_pressure_quantities
if "ne_on_x = self._ne_on_parent_grid" in fd_src and "_set_D_KBM_from_ne_profile(ne_on_x)" in fd_src:
    findings.append(
            "HIGH (if transport update enabled): _update_firedrake_ne_transport_from_ne maps n_e to "
            "x_init then calls calc_pressure_quantities, which expects n_e on x_dofs — length/grid mismatch."
        )

# 2) nCX_x0 — parent solver.py updates it; firedrake only overrides invalidate_firedrake_cache
sp = (ROOT / "src" / "solver.py").read_text()
if "self.nCX_x0 = self.ncx_x0_ratio * self.nFC_x0" in sp:
    findings.append(
            "INFO: nCX_x0 is refreshed in solver.update_free_params when ncx_x0_ratio is passed; "
            "re-instantiating the class is not required."
        )

# 3) _diffusion_coeff_at_inner grid for _D_KBM
if "D_KBM = float(np.interp(x_in, self.x_init, self._D_KBM))" in fd_src:
    findings.append(
            "MEDIUM: _diffusion_coeff_at_inner interpolates _D_KBM from x_init, but "
            "calc_pressure_quantities builds _D_KBM on x_dofs — Neumann D_bc can be wrong."
        )

# 4) Neumann D_bc uses dat.data[0]
if "D_NEO_fd.dat.data[0]" in fd_src:
    findings.append(
            "LOW/MEDIUM: Neumann term uses coefficient at DOF index 0; assumes x_dofs[0]=x_inner. "
            "Usually true for IntervalMesh(x_inner,0) but fragile if ordering changes."
        )

# 5) _interpolate_coefficients_to_mesh vs _D_KBM grid
if "_fill(self._fd_cache[\"D_KBM_fd\"], self._D_KBM)" in fd_src:
    findings.append(
            "MEDIUM: _interpolate_coefficients_to_mesh assumes _D_KBM lives on x_init; "
            "after calc_pressure_quantities it lives on x_dofs (direct assign to D_KBM_fd)."
        )

# 6) Plan A vs A29
findings.append(
    "DOCUMENTATION: Module implements Plan A D_total = D_NEO + D_KBM + C_ETG/n_e, "
    "not appendix A29 expanded weak form with (dn_e/dx)^2 from state-dependent KBM."
)

# 7) Labbate derivation typos (manuscript, not code)
findings.append(
    "TEX (appendix): A29 Neumann line appears to miss '* v_e(x_inner)'; "
    "earlier KBM-strong-form lines have typos in D(ne dn/dx) notation — verify before citing A29 literally."
)

for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")
print(f"\n{len(findings)} findings listed.")

## 6. Grid consistency: `c_s`, `rho_s`, `D_NEO`, `x_init`

After `saarelma_connor.__init__`, transport arrays on the **pressure grid** (`len(psi_pres)`) must match `len(x_init)` for `np.interp(x, x_init, arr)`.

In [ ]:
INPUT_DIR = ROOT / "src" / "inputs" / "PT_Hmode"
MHD_FP = INPUT_DIR / "g189650.02500"
KPROF_FP = INPUT_DIR / "p189650.02500"

model = saarelma_connor_firedrake(
    P_tot_e=5e6,
    alpha_crit=1.0,
    C_KBM=0.1,
    De_chie_etg=0.5,
    nFC_x0=1e17,
    mhd_fp=str(MHD_FP),
    kprof_fp=str(KPROF_FP),
    psi_N_inner_boundary=0.90,
    verbose=False,
)
model.setup_solver_grids(res=200)

n = len(model.x_init)
checks = {
    "x_init": len(model.x_init),
    "psi_pres": len(model.psi_pres),
    "c_s": len(model.c_s),
    "rho_s": len(model.rho_s),
    "D_NEO": len(model.D_NEO),
    "C_ETG": len(model.C_ETG),
    "S_i_pres": len(model.S_i_pres),
}
print(checks)
for name, L in checks.items():
    assert L == n, f"{name} length {L} != x_init {n}"
assert_array_less(model.x_init[:-1], model.x_init[1:])  # increasing toward separatrix
print("OK: parent-grid arrays aligned with x_init for interp.")

## 7. Firedrake: mesh boundaries and `F1` volume assembly

Expect `IntervalMesh(x_inner, 0)` → `ds(1)` at $x_{\mathrm{inner}}$, `ds(2)` at separatrix.

In [ ]:
if not _HAS_FD:
    print("SKIP: Firedrake not available")
else:
    from firedrake import SpatialCoordinate, TestFunctions, MixedFunctionSpace, TrialFunction, split

    x_left, x_right = -0.03, 0.0
    mesh = IntervalMesh(20, x_left, x_right)
    V = FunctionSpace(mesh, "CG", 2)
    W = MixedFunctionSpace([V, V, V])
    x_coord = Function(V).interpolate(SpatialCoordinate(mesh)[0])
    x_dofs = x_coord.dat.data.copy()
    assert x_dofs[0] < x_dofs[-1]
    assert_allclose(x_dofs[0], x_left, rtol=1e-10, atol=1e-12)
    assert_allclose(x_dofs[-1], x_right, rtol=1e-10, atol=1e-12)
    print("OK: mesh coords run from x_inner to separatrix.")

    # Instantiate minimal model piece for _build_f1_weak_form
    m = model
    m._fd_cache["x_dofs"] = x_dofs
    g = Function(V).assign(1.2)
    Dneo = Function(V).assign(0.3)
    Dkbm = Function(V).assign(0.1)
    Cetg = Function(V).assign(1e18)
    Si = Function(V).assign(2e-20)
    u = Function(W)
    for i, val in enumerate([2e19, 1e17, 1e16]):
        u.subfunctions[i].assign(val)
    v_e, _, _ = TestFunctions(W)
    ne_t = TrialFunction(V)
    F1 = m._build_f1_weak_form(
        ne_t, u.subfunctions[0],
        g, Cetg, Dneo, Dkbm,
        Si, u.subfunctions[1], u.subfunctions[2],
        v_e, "dirichlet", Constant(0.0),
    )
    print("F1 assembled:", type(F1).__name__)
    print("OK: _build_f1_weak_form runs without error.")

## 8. Firedrake solve smoke test (optional, slow)

Uncomment or run if Firedrake + equilibrium data are available.

In [ ]:
RUN_SOLVE = False  # set True to run full solve

if not _HAS_FD:
    print("SKIP: Firedrake not available")
elif not RUN_SOLVE:
    print("SKIP: set RUN_SOLVE=True to execute solve_firedrake")
else:
    m = saarelma_connor_firedrake(
        P_tot_e=5e6,
        alpha_crit=1.0,
        C_KBM=0.1,
        De_chie_etg=0.5,
        nFC_x0=1e17,
        mhd_fp=str(MHD_FP),
        kprof_fp=str(KPROF_FP),
        psi_N_inner_boundary=0.90,
        verbose=True,
    )
    x, ne, nfc, ncx = m.solve_firedrake(
        x_res=80,
        fe_degree=2,
        use_picard=True,
        picard_tol=1e-3,
        max_picard=20,
        ne_inner_bc="dirichlet",
        initial_guess="linear",
        verbose=True,
    )
    assert len(x) == len(ne) == len(nfc) == len(ncx)
    assert np.all(np.isfinite(ne))
    print("OK: solve_firedrake completed.")

## 9. `ncx_x0_ratio` scan invariance (if ratio passed to `update_free_params`)

Expect **different** `nCX` if `ncx_x0_ratio` changes; if profiles are identical, `nCX_x0` was not updated.

In [ ]:
if not _HAS_FD:
    print("SKIP: Firedrake not available")
else:
    ratios = [0.05, 0.5]
    ncx_peaks = []
    for r in ratios:
        m = saarelma_connor_firedrake(
            P_tot_e=5e6,
            alpha_crit=1.0,
            C_KBM=0.1,
            De_chie_etg=0.5,
            nFC_x0=1e17,
            ncx_x0_ratio=r,
            mhd_fp=str(MHD_FP),
            kprof_fp=str(KPROF_FP),
            psi_N_inner_boundary=0.90,
            verbose=False,
        )
        print(f"ratio={r}: nCX_x0={m.nCX_x0:.3e}")
        _, _, ncx = m.solve_firedrake(
            x_res=60, fe_degree=1, use_picard=True, max_picard=8,
            ne_inner_bc="dirichlet", initial_guess="linear", verbose=False,
        )
        ncx_peaks.append(ncx.max())
    if abs(ncx_peaks[0] - ncx_peaks[1]) < 0.01 * max(ncx_peaks):
        print("WARNING: nCX peak almost unchanged — nCX_x0 likely not applied in BC.")
    else:
        print("OK: nCX responds to ncx_x0_ratio.")

## 10. Summary

**Matches Labbate Eqs. (8)–(10) / weak-nfc / weak-ncx:**
- Coupled structure and sign of `F1` volume terms vs IBP of Eq. (8).
- Strong-form Galerkin for `F2`, `F3` (no IBP on neutral fluxes).
- Dirichlet BCs at separatrix for all three fields.

**Does not match appendix A29 as written:**
- Code uses **Plan A** effective $D$, not expanded $(\partial_x n_e)^2$ weak form.
- $D_{\mathrm{KBM}}$ from Saarelma $\alpha$ (pedestal-averaged in Firedrake path), not $D_{\mathrm{KBM}}=C_{\mathrm{KBM}}\,\partial_x(n_e T_{tot})$ inside the flux.

**Review issues to fix in code (when you choose to edit):**
1. Call transport update inside Picard loop (and pass `n_e` on `x_dofs`, not `x_init`) if $D_{\mathrm{KBM}}$ should follow $n_e$.
2. Align `_D_KBM` / `_diffusion_coeff_at_inner` / `_interpolate_coefficients_to_mesh._fill` on one grid.
3. Neumann `D_bc`: consider `_diffusion_coeff_at_inner` with lagged $n_e$ instead of `self.ne_inner` + frozen `D_*_fd[0]`.
4. Clarify manuscript vs implementation (Plan A vs A29) in thesis text.